In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

os.chdir("/Users/larsboecking/Documents/01_Code/datacentric-ai")

from src.utils.resultHandler import ResultHandler
import matplotlib.pyplot as plt
from src.utils.configHandler import ConfigHandler

from notebooks.src.model_utils import fit_ols_model, fit_mixed_effects_model
from notebooks.src.preprocessing_utils import select_strategy_subset_results


# Load matplotlib style if available
style_path = os.path.join(os.getcwd(), "configs", "visualisations.mplstyle")
if os.path.exists(style_path):
    plt.style.use(style_path)

_config_handler = ConfigHandler()
rh = ResultHandler(_config_handler.SUMMARY_FILE)

folders = rh.get_result_folders()
available_types = rh.summary_df["type"].unique()
dataset_names = rh.summary_df["dataset"].unique()

2026-03-19 14:36:10,017 - INFO - src.utils.configHandler - Initializing ConfigHandler
2026-03-19 14:36:10,024 - INFO - src.utils.resultHandler - Parsed strategy_params into columns: ['p']
2026-03-19 14:36:10,025 - INFO - src.utils.resultHandler - Initialized ResultHandler with summary_csv_path: results/summary.csv


In [2]:
selected_type = "label_quality"
selected_mode = "flip"

model_df = select_strategy_subset_results(selected_type, selected_mode, rh, _config_handler)

2026-03-19 14:36:10,032 - INFO - src.data_handling.datasetHandler - Initialized UCRDataset with name: Beef, path: Univariate_ts
2026-03-19 14:36:10,033 - INFO - src.data_handling.datasetHandler - Loading dataset: Beef
2026-03-19 14:36:10,046 - INFO - src.data_handling.datasetHandler - Dataset Beef loaded successfully
2026-03-19 14:36:10,047 - INFO - src.data_handling.datasetHandler - Initialized UCRDataset with name: GunPoint, path: Univariate_ts
2026-03-19 14:36:10,047 - INFO - src.data_handling.datasetHandler - Loading dataset: GunPoint
2026-03-19 14:36:10,058 - INFO - src.data_handling.datasetHandler - Dataset GunPoint loaded successfully
2026-03-19 14:36:10,058 - INFO - src.data_handling.datasetHandler - Initialized UCRDataset with name: Yoga, path: Univariate_ts
2026-03-19 14:36:10,059 - INFO - src.data_handling.datasetHandler - Loading dataset: Yoga


Loading metadata for 6 datasets...


2026-03-19 14:36:10,242 - INFO - src.data_handling.datasetHandler - Dataset Yoga loaded successfully
2026-03-19 14:36:10,242 - INFO - src.data_handling.datasetHandler - Initialized UCRDataset with name: AsphaltPavementType, path: Univariate_ts
2026-03-19 14:36:10,243 - INFO - src.data_handling.datasetHandler - Loading dataset: AsphaltPavementType
2026-03-19 14:36:11,489 - INFO - src.data_handling.datasetHandler - Dataset AsphaltPavementType loaded successfully
2026-03-19 14:36:11,489 - INFO - src.data_handling.datasetHandler - Initialized UCRDataset with name: BeetleFly, path: Univariate_ts
2026-03-19 14:36:11,490 - INFO - src.data_handling.datasetHandler - Loading dataset: BeetleFly
2026-03-19 14:36:11,498 - INFO - src.data_handling.datasetHandler - Dataset BeetleFly loaded successfully
2026-03-19 14:36:11,498 - INFO - src.data_handling.datasetHandler - Initialized UCRDataset with name: Blink, path: Univariate_ts
2026-03-19 14:36:11,499 - INFO - src.data_handling.datasetHandler - Load

Could not load Blink: 'list' object has no attribute 'shape'


## H1: Baseline Degradation Effect

Test whether accuracy is affected by the degradation parameter `p` alone, without considering dataset characteristics. This establishes the baseline effect size.

In [3]:
h1_result = fit_ols_model(model_df, "accuracy ~ p")
if h1_result:
    print(h1_result['model'].summary())
else:
    print("H1 fit failed")

                            OLS Regression Results                            
Dep. Variable:               accuracy   R-squared:                       0.840
Model:                            OLS   Adj. R-squared:                  0.839
Method:                 Least Squares   F-statistic:                     933.6
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           1.03e-72
Time:                        14:36:11   Log-Likelihood:                 153.60
No. Observations:                 180   AIC:                            -303.2
Df Residuals:                     178   BIC:                            -296.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.8682      0.017     52.018      0.0

## H2: Dataset Characteristics as Confounders

Add all dataset characteristics (N, T, K) to test whether variation in accuracy is driven by problem characteristics rather than degradation itself. Controls for confounding and establishes generalization baseline.

In [4]:
h2_result = fit_ols_model(model_df, "accuracy ~ p + log_N_std + log_T_std + K_std")
if h2_result:
    print(h2_result['model'].summary())
else:
    print("H2 fit failed")

                            OLS Regression Results                            
Dep. Variable:               accuracy   R-squared:                       0.882
Model:                            OLS   Adj. R-squared:                  0.880
Method:                 Least Squares   F-statistic:                     437.8
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           2.39e-81
Time:                        14:36:11   Log-Likelihood:                 180.95
No. Observations:                 180   AIC:                            -353.9
Df Residuals:                     176   BIC:                            -341.1
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.8682      0.014     60.214      0.0

## H3: Degradation-Characteristic Interactions

Test whether the effect of degradation depends on dataset characteristics. E.g., does degradation hurt small datasets more than large ones? Do short sequences degrade differently than long ones? Interaction terms reveal heterogeneous sensitivity to degradation.

In [5]:
h3_result = fit_ols_model(model_df, "accuracy ~  p + log_N_std + log_T_std + K_std + p * log_N_std + p * log_T_std + p * K_std")
if h3_result:
    print(h3_result['model'].summary())
else:
    print("H3 fit failed")

                            OLS Regression Results                            
Dep. Variable:               accuracy   R-squared:                       0.929
Model:                            OLS   Adj. R-squared:                  0.927
Method:                 Least Squares   F-statistic:                     454.6
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           7.37e-98
Time:                        14:36:12   Log-Likelihood:                 226.66
No. Observations:                 180   AIC:                            -441.3
Df Residuals:                     174   BIC:                            -422.2
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       0.8682      0.011     77.176      

## H4: Classifier Type as Fixed Effect

Introduce classifier type as a fixed effect to test whether different classifiers have systematic accuracy differences. This tests whether classifier choice confounds the degradation effect.

In [6]:
h4_result = fit_ols_model(model_df, "accuracy ~ C(classifier) + p + log_N_std + log_T_std + K_std + p * log_N_std + p * log_T_std + p * K_std")
if h4_result:
    print(h4_result['model'].summary())
else:
    print("H4 fit failed")

                            OLS Regression Results                            
Dep. Variable:               accuracy   R-squared:                       0.929
Model:                            OLS   Adj. R-squared:                  0.927
Method:                 Least Squares   F-statistic:                     377.6
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           1.34e-96
Time:                        14:36:12   Log-Likelihood:                 226.88
No. Observations:                 180   AIC:                            -439.8
Df Residuals:                     173   BIC:                            -417.4
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

## H5: Degradation-Characteristic Interactions Controlling for Classifier

Test two-way interactions between degradation and meta-features (p × N, p × T, p × K) while controlling for classifier type as a fixed effect. This reveals whether degradation sensitivity differs by dataset characteristics, applied equally across all classifiers (no classifier-specific coefficients for the interactions).


In [7]:
h5_result = fit_ols_model(model_df, "accuracy ~ p + log_N_std + log_T_std + K_std + C(classifier) + p:log_N_std + p:log_T_std + p:K_std")
if h5_result:
    print(h5_result['model'].summary())
else:
    print("H5 fit failed")


                            OLS Regression Results                            
Dep. Variable:               accuracy   R-squared:                       0.929
Model:                            OLS   Adj. R-squared:                  0.927
Method:                 Least Squares   F-statistic:                     377.6
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           1.34e-96
Time:                        14:36:12   Log-Likelihood:                 226.88
No. Observations:                 180   AIC:                            -439.8
Df Residuals:                     173   BIC:                            -417.4
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

## H6: Meta-Feature × Degradation × Classifier Interactions

Test whether each classifier responds differently to degradation across different dataset characteristics. Models separate p coefficients for each meta-feature within each classifier type. Creates a classifier-specific degradation sensitivity profile: how does each classifier's accuracy degrade as a function of both dataset size (N) and number of classes (K)?

In [8]:
h7_result = fit_ols_model(model_df, "accuracy ~ p + log_N_std + log_T_std + K_std + C(classifier) + + p:log_N_std + p:log_T_std + p:K_std + p:log_N_std:C(classifier) + p:log_T_std:C(classifier) + p:K_std:C(classifier)")
if h7_result:
    print(h7_result['model'].summary())
else:
    print("H7 fit failed")


                            OLS Regression Results                            
Dep. Variable:               accuracy   R-squared:                       0.930
Model:                            OLS   Adj. R-squared:                  0.927
Method:                 Least Squares   F-statistic:                     285.6
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           1.04e-94
Time:                        14:36:12   Log-Likelihood:                 228.55
No. Observations:                 180   AIC:                            -439.1
Df Residuals:                     171   BIC:                            -410.4
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

In [9]:
model_df.columns

Index(['dataset', 'classifier', 'strategy', 'strategy_mode', 'strategy_params',
       'random_seed', 'accuracy', 'f1_score', 'folder', 'type', 'mode', 'p',
       'N_train', 'T', 'K', 'classifier_type', 'log_N', 'log_T', 'p_std',
       'log_N_std', 'log_T_std', 'K_std'],
      dtype='object')

In [10]:
model_df["classifier"].unique()

array(['mini-rocket', 'catch22'], dtype=object)